# Capstone build --- Chapter 8: Planning

The minimal agent of Chapter~1 proposed one tool then finished. The complaint agent runs a longer, fixed sequence: classify the message, extract its facts, search the governing policy, flag regulatory risk, then either draft a reply or escalate. Chapter~8 makes that sequence a plan --- an ordered set of steps, each naming the action it expects and what it produces --- and shows the capstone's agent is that plan made executable.

## The workflow as an ordered plan

A `Plan` is a list of `PlanStep`s. Each step names an action hint, the type of output it is expected to produce and the steps it requires. A `WorkflowPlanner` wraps a fixed ordered plan and returns it for any task, which is the right planner for a workflow that does not vary from case to case.

In [ ]:
from agentlab.planning.plan import Plan, PlanStep
from agentlab.planning.planner import WorkflowPlanner
from agentlab.core.task import TaskSpec

steps = [
    PlanStep(id='classify',        description='classify the message',        action_hint='classify_complaint'),
    PlanStep(id='extract',         description='extract grounded facts',       action_hint='extract_facts',   requires=('classify',)),
    PlanStep(id='search_policy',   description='retrieve governing policy',     action_hint='search_policy',   requires=('extract',)),
    PlanStep(id='flag_regulatory', description='flag regulatory risk',          action_hint='flag_regulatory', requires=('extract',)),
    PlanStep(id='draft_response',  description='draft the reply or escalate',   action_hint='draft_response',  requires=('search_policy', 'flag_regulatory')),
]
planner = WorkflowPlanner(steps)
plan = planner.plan(TaskSpec(goal='handle a complaint', inputs={}))
for i, s in enumerate(plan.steps):
    print(f'{i}: {s.id:16s} -> {s.action_hint:18s} requires={s.requires}')

## The agent is the plan made executable

The capstone hard-codes this fixed order in `ComplaintAgent.propose_action`, a function from state to the next action. It proposes the tool for the current step index, reuses earlier results rather than recomputing them, and diverts to an escalation when a prior step failed or a regulatory flag demands it. Instantiating the agent and stepping a state through it shows the plan running.

In [ ]:
from agentlab.capstone.complaint_agent import ComplaintAgent
from agentlab.core.state import AgentState

agent = ComplaintAgent()
task = TaskSpec(goal='handle a complaint',
                inputs={'message': 'I was charged a $35 overdraft fee I did not authorize.'})
state = AgentState(task=task)
# With no tool results yet, the first proposed action is the plan's first step.
first = agent.propose_action(state)
print('step 0 proposes:', first.kind, '->', getattr(first, 'tool_name', None))

## The plan diverts to escalation on failure

The plan is not a straight line. Before proposing the next step the agent scans the results so far, and a failed tool result --- the shape a denied gate returns in Chapter~6 --- makes it propose an `Escalate` instead of continuing. Injecting a failed result into the state shows the diversion.

In [ ]:
state_with_failure = AgentState(task=task)
state_with_failure.tool_results.append({'success': False, 'error': 'denied by PolicyGate: PII'})
diverted = agent.propose_action(state_with_failure)
print('proposes:', diverted.kind, '| reason:', getattr(diverted, 'reason', None))

The fixed workflow is the capstone's plan: an ordered sequence of typed actions with escape hatches to escalation. Chapter~9 gives the agent the memory that lets a later step reuse an earlier step's result, Chapter~10 evaluates the trajectory the plan produces, and Chapter~12 runs the plan inside the governance harness.